# 25. Trend / Momentum特徴量の比較
出典: FX (2).ipynb、セルindex [54]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 54


In [ ]:
# ============================================================
# FX FEATURE TOURNAMENT #1
# BASE vs BASE + TREND / MOMENTUM
#
# 目的:
#   1. 現在のHGBを固定する
#   2. Trend / Momentum特徴量だけ追加する
#   3. Nested Walk-ForwardでBASEと比較する
#   4. 2026年は最終Confirmationとして使用する
#
# 注意:
#   ・モデルのハイパーパラメータは変更しない
#   ・未来情報を特徴量に入れない
#   ・2026年を特徴量選択に使わない
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------
# 0. 基本設定
# ------------------------------------------------------------

RANDOM_STATE = 42

# 15分足 × 2本 = 30分後
HORIZON_BARS = 2

# 売買コスト
BASE_COST_PCT = 0.004

# 閾値候補
THRESHOLDS = [
    0.55,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
]

# Session候補
SESSION_POLICIES = [
    "ALL",
    "EXCLUDE_08_13",
    "UTC_13_24",
    "UTC_21_24",
]

# HGB Champion固定
HGB_PARAMS = dict(
    learning_rate=0.05,
    max_iter=250,
    max_leaf_nodes=15,
    min_samples_leaf=30,
    l2_regularization=1.0,
    random_state=RANDOM_STATE,
)

# Feature選択に使う年
DEVELOPMENT_YEARS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
]

# 完全Holdout
FINAL_HOLDOUT_YEAR = 2026

MIN_VALIDATION_TRADES = 80


# ============================================================
# 1. OHLC DataFrameを自動検出
# ============================================================

def normalize_name(x):
    return str(x).strip().lower().replace(" ", "_")


def find_ohlc_dataframe(namespace):
    """
    Notebook内からOHLC DataFrameを探す。
    barsを最優先。
    """

    preferred_names = [
        "bars",
        "bars15",
        "bars_15m",
        "price15",
        "ohlc",
        "df",
        "data",
    ]

    candidates = []

    for name, obj in namespace.items():

        if not isinstance(obj, pd.DataFrame):
            continue

        if len(obj) < 1000:
            continue

        cols = {
            normalize_name(c): c
            for c in obj.columns
        }

        has_ohlc = all(
            x in cols
            for x in ["open", "high", "low", "close"]
        )

        if not has_ohlc:
            continue

        score = len(obj)

        if name in preferred_names:
            score += 10_000_000

        candidates.append(
            (
                score,
                name,
                obj
            )
        )

    if not candidates:
        raise RuntimeError(
            "\nOHLC DataFrameを見つけられませんでした。\n"
            "open/high/low/closeを持つDataFrameが必要です。"
        )

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    _, name, frame = candidates[0]

    print(
        f"使用DataFrame: {name}"
    )

    return frame.copy()


raw = find_ohlc_dataframe(globals())


# ============================================================
# 2. OHLCを標準化
# ============================================================

def prepare_ohlc(df):

    out = df.copy()

    # -----------------------------
    # Timestamp処理
    # -----------------------------

    timestamp_candidates = [
        "timestamp",
        "datetime",
        "date",
        "time",
    ]

    found_time = None

    for c in out.columns:

        if normalize_name(c) in timestamp_candidates:

            found_time = c
            break

    if found_time is not None:

        out[found_time] = pd.to_datetime(
            out[found_time],
            errors="coerce",
            utc=True
        )

        out = out.dropna(
            subset=[found_time]
        )

        out = out.set_index(
            found_time
        )

    elif not isinstance(
        out.index,
        pd.DatetimeIndex
    ):

        try:

            out.index = pd.to_datetime(
                out.index,
                errors="coerce",
                utc=True
            )

        except Exception:

            raise RuntimeError(
                "timestampをDatetimeへ変換できませんでした。"
            )

    # timezoneが無い場合
    if out.index.tz is None:

        out.index = out.index.tz_localize(
            "UTC"
        )

    # -----------------------------
    # OHLC列名統一
    # -----------------------------

    rename_map = {}

    for c in out.columns:

        n = normalize_name(c)

        if n in [
            "open",
            "high",
            "low",
            "close",
            "volume"
        ]:

            rename_map[c] = n

    out = out.rename(
        columns=rename_map
    )

    required = [
        "open",
        "high",
        "low",
        "close"
    ]

    missing = [
        c
        for c in required
        if c not in out.columns
    ]

    if missing:

        raise RuntimeError(
            f"OHLC不足: {missing}"
        )

    # -----------------------------
    # 数値化
    # -----------------------------

    for c in required:

        out[c] = pd.to_numeric(
            out[c],
            errors="coerce"
        )

    out = (
        out
        .sort_index()
        .loc[
            ~out.index.duplicated(
                keep="last"
            )
        ]
    )

    out = out.replace(
        [np.inf, -np.inf],
        np.nan
    )

    out = out.dropna(
        subset=required
    )

    return out


bars_ft = prepare_ohlc(raw)


print(
    "\n"
    "========================================"
)

print(
    "DATA CHECK"
)

print(
    "========================================"
)

print(
    "Rows:",
    len(bars_ft)
)

print(
    "Start:",
    bars_ft.index.min()
)

print(
    "End:",
    bars_ft.index.max()
)


# ============================================================
# 3. 足間隔チェック
# ============================================================

diff_min = (
    bars_ft.index
    .to_series()
    .diff()
    .dropna()
    .dt.total_seconds()
    .div(60)
)

if len(diff_min) > 0:

    typical_interval = float(
        diff_min.median()
    )

else:

    typical_interval = np.nan


print(
    "Median interval:",
    typical_interval,
    "minutes"
)


if (
    np.isfinite(typical_interval)
    and abs(
        typical_interval - 15
    ) > 2
):

    print(
        "WARNING:"
        " データ間隔が15分から大きく離れています。"
    )


# ============================================================
# 4. Target作成
#
# 現在Close → 30分後Close
# ============================================================

bars_ft["future_close"] = (
    bars_ft["close"]
    .shift(-HORIZON_BARS)
)

bars_ft["forward_return"] = (
    bars_ft["future_close"]
    / bars_ft["close"]
    - 1
)

bars_ft["target"] = (
    bars_ft["forward_return"] > 0
).astype(int)


# ============================================================
# 5. BASE特徴量
#
# 既存のfeature_colsがあれば最優先。
# 無ければOHLCから安全なBaselineを作る。
# ============================================================

def find_existing_feature_list(namespace, df):

    candidate_names = [
        "feature_cols",
        "FEATURE_COLS",
        "features",
        "FEATURES",
        "base_features",
        "BASE_FEATURES",
    ]

    for name in candidate_names:

        if name not in namespace:
            continue

        obj = namespace[name]

        if not isinstance(
            obj,
            (list, tuple)
        ):
            continue

        usable = [
            c
            for c in obj
            if c in df.columns
        ]

        if len(usable) >= 5:

            print(
                f"既存Feature List使用: {name}"
            )

            print(
                f"Features: {len(usable)}"
            )

            return usable

    return None


existing_features = find_existing_feature_list(
    globals(),
    bars_ft
)


# ------------------------------------------------------------
# 既存Featureが無い場合のBaseline
# ------------------------------------------------------------

def add_baseline_features(df):

    x = df.copy()

    close = x["close"]
    high = x["high"]
    low = x["low"]
    open_ = x["open"]

    # Return
    for n in [
        1,
        2,
        4,
        8,
        16,
        32
    ]:

        x[f"ret_{n}"] = (
            close.pct_change(n)
        )

    # Log return
    log_ret = np.log(
        close
    ).diff()

    x["log_ret_1"] = log_ret

    # Rolling volatility
    for n in [
        4,
        8,
        16,
        32,
        64
    ]:

        x[f"vol_{n}"] = (
            log_ret
            .rolling(n)
            .std()
        )

    # Candle構造
    x["candle_return"] = (
        close / open_ - 1
    )

    x["range_pct"] = (
        high / low - 1
    )

    denominator = (
        high - low
    ).replace(
        0,
        np.nan
    )

    x["close_location"] = (
        (
            close - low
        )
        / denominator
    )

    x["body_ratio"] = (
        (
            close - open_
        ).abs()
        / denominator
    )

    x["upper_wick"] = (
        high
        - np.maximum(
            open_,
            close
        )
    ) / close

    x["lower_wick"] = (
        np.minimum(
            open_,
            close
        )
        - low
    ) / close

    # Rolling Z
    for n in [
        8,
        16,
        32,
        64
    ]:

        mean = (
            close
            .rolling(n)
            .mean()
        )

        std = (
            close
            .rolling(n)
            .std()
        )

        x[f"price_z_{n}"] = (
            close - mean
        ) / std

    # 時刻
    x["hour_sin"] = np.sin(
        2
        * np.pi
        * x.index.hour
        / 24
    )

    x["hour_cos"] = np.cos(
        2
        * np.pi
        * x.index.hour
        / 24
    )

    x["dow_sin"] = np.sin(
        2
        * np.pi
        * x.index.dayofweek
        / 7
    )

    x["dow_cos"] = np.cos(
        2
        * np.pi
        * x.index.dayofweek
        / 7
    )

    return x


if existing_features is None:

    bars_ft = add_baseline_features(
        bars_ft
    )

    base_features = [
        c
        for c in bars_ft.columns
        if (
            c.startswith("ret_")
            or c.startswith("vol_")
            or c.startswith("price_z_")
            or c in [
                "log_ret_1",
                "candle_return",
                "range_pct",
                "close_location",
                "body_ratio",
                "upper_wick",
                "lower_wick",
                "hour_sin",
                "hour_cos",
                "dow_sin",
                "dow_cos",
            ]
        )
    ]

else:

    base_features = (
        existing_features.copy()
    )


print(
    "\nBASE features:",
    len(base_features)
)


# ============================================================
# 6. Trend / Momentum特徴量
# ============================================================

def add_trend_momentum_features(df):

    x = df.copy()

    close = x["close"]

    # --------------------------------------------------------
    # EMA
    # --------------------------------------------------------

    ema_periods = [
        4,
        8,
        16,
        32,
        64,
        128,
    ]

    for n in ema_periods:

        ema = (
            close
            .ewm(
                span=n,
                adjust=False
            )
            .mean()
        )

        # 現在値とEMAの距離
        x[f"tm_ema_dist_{n}"] = (
            close / ema - 1
        )

        # EMA slope
        x[f"tm_ema_slope_{n}"] = (
            ema.pct_change(4)
        )

    # --------------------------------------------------------
    # EMA spread
    # --------------------------------------------------------

    ema4 = close.ewm(
        span=4,
        adjust=False
    ).mean()

    ema8 = close.ewm(
        span=8,
        adjust=False
    ).mean()

    ema16 = close.ewm(
        span=16,
        adjust=False
    ).mean()

    ema32 = close.ewm(
        span=32,
        adjust=False
    ).mean()

    ema64 = close.ewm(
        span=64,
        adjust=False
    ).mean()

    x["tm_ema_4_16"] = (
        ema4 / ema16 - 1
    )

    x["tm_ema_8_32"] = (
        ema8 / ema32 - 1
    )

    x["tm_ema_16_64"] = (
        ema16 / ema64 - 1
    )

    # --------------------------------------------------------
    # Momentum
    # --------------------------------------------------------

    for n in [
        2,
        4,
        8,
        12,
        16,
        24,
        32,
        48,
        64,
    ]:

        x[f"tm_momentum_{n}"] = (
            close.pct_change(n)
        )

    # --------------------------------------------------------
    # Momentum acceleration
    # --------------------------------------------------------

    mom4 = close.pct_change(4)
    mom8 = close.pct_change(8)
    mom16 = close.pct_change(16)

    x["tm_momentum_accel_4_8"] = (
        mom4 - mom8
    )

    x["tm_momentum_accel_8_16"] = (
        mom8 - mom16
    )

    # --------------------------------------------------------
    # MACD系
    # --------------------------------------------------------

    ema12 = close.ewm(
        span=12,
        adjust=False
    ).mean()

    ema26 = close.ewm(
        span=26,
        adjust=False
    ).mean()

    macd = (
        ema12 - ema26
    ) / close

    macd_signal = (
        macd
        .ewm(
            span=9,
            adjust=False
        )
        .mean()
    )

    x["tm_macd"] = macd

    x["tm_macd_signal"] = (
        macd_signal
    )

    x["tm_macd_hist"] = (
        macd
        - macd_signal
    )

    # --------------------------------------------------------
    # Trend consistency
    #
    # 最近N本で上昇した割合
    # --------------------------------------------------------

    up_bar = (
        close.diff() > 0
    ).astype(float)

    for n in [
        4,
        8,
        16,
        32,
        64,
    ]:

        x[f"tm_up_ratio_{n}"] = (
            up_bar
            .rolling(n)
            .mean()
        )

    # --------------------------------------------------------
    # Linear trend slope
    # --------------------------------------------------------

    def rolling_slope(arr):

        y = np.asarray(
            arr,
            dtype=float
        )

        n = len(y)

        if n < 2:
            return np.nan

        xx = np.arange(
            n,
            dtype=float
        )

        xx = (
            xx
            - xx.mean()
        )

        yy = (
            y
            - y.mean()
        )

        denominator = np.sum(
            xx ** 2
        )

        if denominator == 0:
            return 0.0

        slope = (
            np.sum(
                xx * yy
            )
            / denominator
        )

        mean_price = np.mean(
            y
        )

        if mean_price == 0:
            return 0.0

        return (
            slope
            / mean_price
        )

    for n in [
        8,
        16,
        32,
        64,
    ]:

        x[f"tm_slope_{n}"] = (
            close
            .rolling(n)
            .apply(
                rolling_slope,
                raw=True
            )
        )

    return x


bars_ft = add_trend_momentum_features(
    bars_ft
)


trend_features = [
    c
    for c in bars_ft.columns
    if c.startswith(
        "tm_"
    )
]


print(
    "Trend/Momentum features:",
    len(trend_features)
)


# ============================================================
# 7. Dataset整理
# ============================================================

all_features = list(
    dict.fromkeys(
        base_features
        + trend_features
    )
)


# leakage候補を除外
leak_keywords = [
    "future",
    "target",
    "label",
    "forward_return",
    "prediction",
    "probability",
    "profit",
    "exit_return",
]


def is_safe_feature(c):

    name = normalize_name(c)

    for word in leak_keywords:

        if word in name:
            return False

    return True


base_features = [
    c
    for c in base_features
    if (
        c in bars_ft.columns
        and is_safe_feature(c)
    )
]


trend_features = [
    c
    for c in trend_features
    if (
        c in bars_ft.columns
        and is_safe_feature(c)
    )
]


feature_sets = {

    "BASE":
        base_features,

    "BASE_PLUS_TREND":
        list(
            dict.fromkeys(
                base_features
                + trend_features
            )
        ),
}


print(
    "\n"
    "========================================"
)

print(
    "FEATURE SETS"
)

print(
    "========================================"
)

for k, v in feature_sets.items():

    print(
        k,
        ":",
        len(v)
    )


# ============================================================
# 8. Session Filter
# ============================================================

def session_mask(index, policy):

    hour = index.hour

    if policy == "ALL":

        return np.ones(
            len(index),
            dtype=bool
        )

    if policy == "EXCLUDE_08_13":

        return ~(
            (hour >= 8)
            & (hour < 13)
        )

    if policy == "UTC_13_24":

        return (
            hour >= 13
        )

    if policy == "UTC_21_24":

        return (
            hour >= 21
        )

    return np.ones(
        len(index),
        dtype=bool
    )


# ============================================================
# 9. Trading return作成
# ============================================================

def make_trade_returns(
    df,
    probability,
    threshold,
    session_policy,
    cost_pct,
):

    confidence = (
        np.abs(
            probability - 0.5
        )
        * 2
    )

    pred_up = (
        probability >= 0.5
    )

    take = (
        confidence >= (
            threshold - 0.5
        ) * 2
    )

    take &= session_mask(
        df.index,
        session_policy
    )

    direction = np.where(
        pred_up,
        1.0,
        -1.0
    )

    gross = (
        direction
        * df["forward_return"].values
    )

    # percent -> decimal
    cost = (
        cost_pct
        / 100.0
    )

    net = (
        gross
        - cost
    )

    result = pd.DataFrame(
        {
            "return": net,
            "gross_return": gross,
            "taken": take,
        },
        index=df.index
    )

    return result.loc[
        result["taken"]
    ].copy()


# ============================================================
# 10. Performance Metrics
# ============================================================

def performance_stats(
    trade_returns
):

    if len(trade_returns) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    r = (
        trade_returns["return"]
        .dropna()
        .values
    )

    if len(r) == 0:

        return {
            "trades": 0,
            "win_rate": np.nan,
            "avg_return": np.nan,
            "profit_factor": np.nan,
            "growth": np.nan,
            "max_dd": np.nan,
            "return_to_dd": np.nan,
        }

    wins = r[
        r > 0
    ]

    losses = r[
        r < 0
    ]

    gross_profit = (
        wins.sum()
        if len(wins)
        else 0.0
    )

    gross_loss = (
        -losses.sum()
        if len(losses)
        else 0.0
    )

    if gross_loss > 0:

        pf = (
            gross_profit
            / gross_loss
        )

    elif gross_profit > 0:

        pf = np.inf

    else:

        pf = np.nan

    equity = np.cumprod(
        1.0 + r
    )

    peak = np.maximum.accumulate(
        equity
    )

    dd = (
        equity
        / peak
        - 1
    )

    max_dd = (
        dd.min()
        if len(dd)
        else 0
    )

    growth = (
        equity[-1]
        - 1
    )

    if max_dd < 0:

        return_to_dd = (
            growth
            / abs(
                max_dd
            )
        )

    else:

        return_to_dd = np.nan

    return {

        "trades":
            len(r),

        "win_rate":
            np.mean(
                r > 0
            ),

        "avg_return":
            np.mean(r),

        "profit_factor":
            pf,

        "growth":
            growth,

        "max_dd":
            max_dd,

        "return_to_dd":
            return_to_dd,
    }


# ============================================================
# 11. ValidationでThreshold / Sessionを選択
# ============================================================

def choose_policy(
    validation_df,
    probability,
    cost_pct,
):

    rows = []

    for threshold in THRESHOLDS:

        for session in SESSION_POLICIES:

            trades = make_trade_returns(
                validation_df,
                probability,
                threshold,
                session,
                cost_pct,
            )

            s = performance_stats(
                trades
            )

            if (
                s["trades"]
                < MIN_VALIDATION_TRADES
            ):
                continue

            pf = s["profit_factor"]

            if not np.isfinite(pf):
                pf = 10.0

            # -----------------------------------------
            # Selection score
            #
            # PF + Return/DD + AvgReturn
            # -----------------------------------------

            score = (
                0.45
                * np.clip(
                    pf,
                    0,
                    3
                )
                / 3

                + 0.35
                * np.clip(
                    s["return_to_dd"],
                    -2,
                    10
                )
                / 10

                + 0.20
                * np.clip(
                    s["avg_return"]
                    * 10000,
                    -5,
                    5
                )
                / 5
            )

            rows.append(
                {
                    "threshold":
                        threshold,

                    "session":
                        session,

                    "score":
                        score,

                    **s,
                }
            )

    if len(rows) == 0:

        return {
            "threshold":
                0.58,

            "session":
                "ALL",
        }

    table = pd.DataFrame(
        rows
    )

    table = table.sort_values(
        "score",
        ascending=False
    )

    best = table.iloc[0]

    return {
        "threshold":
            float(
                best["threshold"]
            ),

        "session":
            str(
                best["session"]
            ),
    }


# ============================================================
# 12. 1 Feature SetをWalk-Forward評価
# ============================================================

def evaluate_feature_set(
    df,
    features,
    feature_set_name,
):

    clean = df.copy()

    required = (
        features
        + [
            "target",
            "forward_return"
        ]
    )

    clean = clean.replace(
        [np.inf, -np.inf],
        np.nan
    )

    clean = clean.dropna(
        subset=required
    )

    years = clean.index.year

    annual_rows = []

    trade_frames = []

    print(
        "\n"
        "========================================"
    )

    print(
        feature_set_name
    )

    print(
        "========================================"
    )

    for test_year in (
        DEVELOPMENT_YEARS
        + [
            FINAL_HOLDOUT_YEAR
        ]
    ):

        validation_year = (
            test_year - 1
        )

        train_mask = (
            years
            <= test_year - 2
        )

        val_mask = (
            years
            == validation_year
        )

        test_mask = (
            years
            == test_year
        )

        train_df = clean.loc[
            train_mask
        ]

        val_df = clean.loc[
            val_mask
        ]

        test_df = clean.loc[
            test_mask
        ]

        if (
            len(train_df) < 5000
            or len(val_df) < 500
            or len(test_df) < 500
        ):

            print(
                f"{test_year}: "
                "データ不足のためskip"
            )

            continue

        X_train = (
            train_df[features]
            .astype(float)
        )

        y_train = (
            train_df["target"]
            .astype(int)
        )

        X_val = (
            val_df[features]
            .astype(float)
        )

        X_test = (
            test_df[features]
            .astype(float)
        )

        y_test = (
            test_df["target"]
            .astype(int)
        )

        # -----------------------------------------
        # HGB
        # -----------------------------------------

        model = (
            HistGradientBoostingClassifier(
                **HGB_PARAMS
            )
        )

        model.fit(
            X_train,
            y_train
        )

        val_prob = (
            model.predict_proba(
                X_val
            )[:, 1]
        )

        test_prob = (
            model.predict_proba(
                X_test
            )[:, 1]
        )

        # -----------------------------------------
        # ValidationでThreshold + Session選択
        # -----------------------------------------

        policy = choose_policy(
            val_df,
            val_prob,
            BASE_COST_PCT,
        )

        threshold = (
            policy["threshold"]
        )

        session = (
            policy["session"]
        )

        # -----------------------------------------
        # AUC
        # -----------------------------------------

        try:

            auc = roc_auc_score(
                y_test,
                test_prob
            )

        except Exception:

            auc = np.nan

        # -----------------------------------------
        # Test
        # -----------------------------------------

        trades = make_trade_returns(
            test_df,
            test_prob,
            threshold,
            session,
            BASE_COST_PCT,
        )

        stats = performance_stats(
            trades
        )

        trades = trades.copy()

        trades["feature_set"] = (
            feature_set_name
        )

        trades["test_year"] = (
            test_year
        )

        trade_frames.append(
            trades
        )

        row = {

            "feature_set":
                feature_set_name,

            "test_year":
                test_year,

            "validation_year":
                validation_year,

            "threshold":
                threshold,

            "session":
                session,

            "auc":
                auc,

            **stats,
        }

        annual_rows.append(
            row
        )

        print(
            "\n"
            f"TEST YEAR {test_year}"
        )

        print(
            f"AUC       : {auc:.4f}"
        )

        print(
            f"Threshold : {threshold}"
        )

        print(
            f"Session   : {session}"
        )

        print(
            f"Trades    : {stats['trades']}"
        )

        print(
            f"Win       : "
            f"{stats['win_rate']*100:.2f}%"
        )

        print(
            f"Avg Return: "
            f"{stats['avg_return']*100:.5f}%"
        )

        print(
            f"PF        : "
            f"{stats['profit_factor']:.3f}"
        )

        print(
            f"Growth    : "
            f"{stats['growth']*100:.3f}%"
        )

        print(
            f"Max DD    : "
            f"{stats['max_dd']*100:.3f}%"
        )

    annual = pd.DataFrame(
        annual_rows
    )

    if len(trade_frames):

        all_trades = pd.concat(
            trade_frames
        )

    else:

        all_trades = pd.DataFrame()

    return (
        annual,
        all_trades,
    )


# ============================================================
# 13. Tournament実行
# ============================================================

all_annual = []

all_trades = []


for feature_set_name, features in feature_sets.items():

    annual, trades = (
        evaluate_feature_set(
            bars_ft,
            features,
            feature_set_name,
        )
    )

    all_annual.append(
        annual
    )

    all_trades.append(
        trades
    )


annual_results = pd.concat(
    all_annual,
    ignore_index=True
)

trade_results = pd.concat(
    all_trades,
    ignore_index=False
)


# ============================================================
# 14. Development 2020-2025比較
# ============================================================

development = annual_results[
    annual_results["test_year"]
    .isin(
        DEVELOPMENT_YEARS
    )
].copy()


summary_rows = []


for feature_set, group in development.groupby(
    "feature_set"
):

    trades = trade_results[
        (
            trade_results["feature_set"]
            == feature_set
        )
        &
        (
            trade_results["test_year"]
            .isin(
                DEVELOPMENT_YEARS
            )
        )
    ]

    stats = performance_stats(
        trades
    )

    positive_years = int(
        (
            group["avg_return"]
            > 0
        ).sum()
    )

    pf_years = int(
        (
            group["profit_factor"]
            > 1
        ).sum()
    )

    summary_rows.append(
        {
            "feature_set":
                feature_set,

            "years":
                len(group),

            "mean_auc":
                group["auc"].mean(),

            "positive_years":
                positive_years,

            "pf_above_1_years":
                pf_years,

            **stats,
        }
    )


development_summary = pd.DataFrame(
    summary_rows
)


print(
    "\n\n"
    "========================================"
)

print(
    "DEVELOPMENT SUMMARY 2020-2025"
)

print(
    "========================================"
)

print(
    development_summary
    .sort_values(
        "profit_factor",
        ascending=False
    )
    .to_string(
        index=False
    )
)


# ============================================================
# 15. 2026 Confirmation
# ============================================================

confirmation = annual_results[
    annual_results["test_year"]
    == FINAL_HOLDOUT_YEAR
].copy()


print(
    "\n\n"
    "========================================"
)

print(
    "FINAL CONFIRMATION 2026"
)

print(
    "========================================"
)

print(
    confirmation[
        [
            "feature_set",
            "auc",
            "trades",
            "win_rate",
            "avg_return",
            "profit_factor",
            "growth",
            "max_dd",
            "return_to_dd",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 16. COST STRESS
# ============================================================

print(
    "\n\n"
    "========================================"
)

print(
    "COST STRESS"
)

print(
    "========================================"
)


cost_rows = []


for feature_set in feature_sets.keys():

    subset = trade_results[
        (
            trade_results["feature_set"]
            == feature_set
        )
        &
        (
            trade_results["test_year"]
            .isin(
                DEVELOPMENT_YEARS
            )
        )
    ].copy()

    if len(subset) == 0:
        continue

    # trade_resultsのreturnは既に1xコスト控除済み
    gross = (
        subset["gross_return"]
        .values
    )

    for cost_x in [
        1.0,
        1.5,
        2.0
    ]:

        cost_decimal = (
            BASE_COST_PCT
            * cost_x
            / 100
        )

        temp = pd.DataFrame(
            {
                "return":
                    gross
                    - cost_decimal
            }
        )

        s = performance_stats(
            temp
        )

        cost_rows.append(
            {
                "feature_set":
                    feature_set,

                "cost_x":
                    cost_x,

                "cost_pct":
                    BASE_COST_PCT
                    * cost_x,

                **s,
            }
        )


cost_stress = pd.DataFrame(
    cost_rows
)


print(
    cost_stress[
        [
            "feature_set",
            "cost_x",
            "trades",
            "avg_return",
            "profit_factor",
            "growth",
            "max_dd",
            "return_to_dd",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 17. 自動判定
# ============================================================

print(
    "\n\n"
    "========================================"
)

print(
    "AUTOMATIC FEATURE DECISION"
)

print(
    "========================================"
)


base_row = development_summary[
    development_summary["feature_set"]
    == "BASE"
]

trend_row = development_summary[
    development_summary["feature_set"]
    == "BASE_PLUS_TREND"
]


if (
    len(base_row)
    == 1
    and len(trend_row)
    == 1
):

    base_row = base_row.iloc[0]
    trend_row = trend_row.iloc[0]

    checks = {}

    checks["AUC"] = (
        trend_row["mean_auc"]
        >= base_row["mean_auc"]
    )

    checks["PF"] = (
        trend_row["profit_factor"]
        > base_row["profit_factor"]
    )

    checks["AVG_RETURN"] = (
        trend_row["avg_return"]
        > base_row["avg_return"]
    )

    checks["RETURN_DD"] = (
        trend_row["return_to_dd"]
        > base_row["return_to_dd"]
    )

    checks["POSITIVE_YEARS"] = (
        trend_row["positive_years"]
        >= base_row["positive_years"]
    )

    wins = sum(
        checks.values()
    )

    print(
        "Metric wins:",
        wins,
        "/",
        len(checks)
    )

    for k, v in checks.items():

        print(
            f"{k:15s}:",
            v
        )

    # 2026確認
    base_2026 = confirmation[
        confirmation["feature_set"]
        == "BASE"
    ]

    trend_2026 = confirmation[
        confirmation["feature_set"]
        == "BASE_PLUS_TREND"
    ]

    confirmation_good = False

    if (
        len(base_2026) == 1
        and len(trend_2026) == 1
    ):

        b = base_2026.iloc[0]
        t = trend_2026.iloc[0]

        confirmation_good = (
            (
                t["profit_factor"]
                >= b["profit_factor"]
            )
            or
            (
                t["avg_return"]
                >= b["avg_return"]
            )
        )

    # cost 2x
    trend_cost2 = cost_stress[
        (
            cost_stress["feature_set"]
            == "BASE_PLUS_TREND"
        )
        &
        (
            cost_stress["cost_x"]
            == 2.0
        )
    ]

    cost_good = False

    if len(trend_cost2) == 1:

        cost_good = (
            trend_cost2.iloc[0][
                "profit_factor"
            ]
            > 1
        )

    print(
        "\n2026 confirmation:",
        confirmation_good
    )

    print(
        "2x cost PF > 1:",
        cost_good
    )

    print(
        "\n"
        "----------------------------------------"
    )

    if (
        wins >= 3
        and confirmation_good
        and cost_good
    ):

        print(
            "RESULT:"
        )

        print(
            "TREND / MOMENTUM FEATURES "
            "SHOW OOS VALUE"
        )

        print(
            "\n次へ:"
        )

        print(
            "Trend/Momentumを候補として保持し、"
            "Volatility / Regime特徴量Tournamentへ進む。"
        )

    else:

        print(
            "RESULT:"
        )

        print(
            "TREND / MOMENTUM VALUE "
            "IS NOT YET CLEAR"
        )

        print(
            "\n次へ:"
        )

        print(
            "Trend/MomentumはBASEへまだ統合せず、"
            "Volatility / Regime特徴量を別個に検証する。"
        )


# ============================================================
# 18. 結果をNotebookに残す
# ============================================================

FEATURE_TOURNAMENT_1_ANNUAL = (
    annual_results.copy()
)

FEATURE_TOURNAMENT_1_SUMMARY = (
    development_summary.copy()
)

FEATURE_TOURNAMENT_1_CONFIRMATION = (
    confirmation.copy()
)

FEATURE_TOURNAMENT_1_COST = (
    cost_stress.copy()
)


print(
    "\n\n"
    "========================================"
)

print(
    "FINISHED"
)

print(
    "========================================"
)

print(
    "保存されたNotebook変数:"
)

print(
    "FEATURE_TOURNAMENT_1_ANNUAL"
)

print(
    "FEATURE_TOURNAMENT_1_SUMMARY"
)

print(
    "FEATURE_TOURNAMENT_1_CONFIRMATION"
)

print(
    "FEATURE_TOURNAMENT_1_COST"
)
